# 13. Face Embeddings — one synthetic face per customer as a feature-set (FOC-178, phase F4)

The F4 question, taken to its deliberately extreme end: **does a per-customer
FACE — one StyleGAN2 image synthesized deterministically from the customer id
and embedded by a pretrained FaceNet — add fraud signal beyond xgb-client,
under the identical runner protocol?**

**Pre-registration (the F4 plan §7), stated BEFORE any result below:**
appearance carries NO fraud signal. The honest scientific reading is that a
per-customer face embedding is at best a customer-identity proxy:

- on the two customer-grouped axes (random-grouped PRIMARY, grouped) a test
  customer was never seen in train, so its face is an arbitrary unseen
  constant — no transfer is possible and the pre-registered expectation is a
  chance-level null;
- on the chronological axis a test customer usually WAS seen in train, so the
  shared face is an identity channel — any lift there is leakage by design
  (the identity memorization nb7 already demonstrated with client features),
  a statement about identity transfer, never about appearance.

We test it anyway and report exactly what we see.

**Determinism (decision D5), exact rule:** the latent seed for customer C is
`sha256(("face-arm-v1:" + C).encode("utf-8"))` — first 8 bytes read as a
big-endian integer; `z = numpy.random.RandomState(seed).randn(1, z_dim)`;
`w = G.mapping(z, zero label, truncation_psi=0.7)`; synthesis with
`noise_mode='const'`, `force_fp32=True` (the pkl's own const noise buffers —
no RNG is consumed at synthesis time). Same customer_id ⇒ same z ⇒ same
pixels ⇒ same JPEG bytes ⇒ same 512-d FaceNet embedding, reproducible on
this machine. The generation cache lives in `data/faces/` (JPEG, 512 px long
edge, <= 50 MB) and the committed embedding artifact in
`data/face_embeddings.npz` (100 x 512 float32 + customer_id), so downstream
consumers (nb15, the runner) never need a GPU pass.

**No network fetch anywhere** — the dynamic faces service
thispersondoesnotexist was rejected up front as non-reproducible; the
StyleGAN2 repo (`C:/sg2-ada`) and the FFHQ weights (`C:/faces/ffhq.pkl`) are
machine-local, the FaceNet vggface2 checkpoint is already in the local torch
cache, and every load below is local-only.

An ethical caveat has its own dedicated cell below the result tables —
face-to-fraud scoring is ethically dubious and this arm exists as a
methods-comparability experiment with a pre-registered null, not a
recommendation.

In [ ]:
# Runtime provenance - executed in the phase worktree venv (kernel python3).
# Printed so the committed, executed notebook self-documents the exact
# runtime the numbers were produced on.
import platform
import sys

import facenet_pytorch
import numpy
import pandas
import sklearn
import torch

import arms_face

print('python:', sys.version.split()[0], '| platform:', platform.platform())
print('kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)')
for _mod in (pandas, numpy, sklearn, torch):
    print('%s: %s' % (_mod.__name__, _mod.__version__))
print('facenet_pytorch: %s' % getattr(facenet_pytorch, '__version__', 'n/a'))
print('torch cuda available:', torch.cuda.is_available())
print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')
print('stylegan2 repo:', arms_face.STYLEGAN_REPO_DIR, '->', arms_face.STYLEGAN_REPO_DIR.exists())
print('ffhq weights:', arms_face.STYLEGAN_WEIGHTS_PATH, '->', arms_face.STYLEGAN_WEIGHTS_PATH.exists())

## The face cache — 100 synthetic faces, generate-or-load

`data/faces/<customer_id>.jpg`, one canonical image per customer, JPEG
quality 90, long edge 512 px, whole directory asserted <= 50 MB. The 512 px
downscale from the 1024 px synthesis is LOSSLESS FOR THIS PURPOSE, not
bit-lossless vs full resolution: FaceNet consumes a 160 x 160 resize, so
detail above the cache resolution cannot influence the 512-d embedding —
while 1024 px JPEGs would quadruple the committed cache for nothing.

Generation is chunkable by construction: `arms_face.ensure_face_image(c)` is
the per-customer unit (generate-or-load, one whole file at a time), so a
killed cell never poisons the cache — the next run continues with whatever
already exists, and a 600 s ceiling can at worst leave a valid partial cache.

FaceNet preprocessing (the standard facenet-pytorch recipe): cache JPEG ->
RGB -> resize 160 x 160 (LANCZOS) -> /255 -> (x - 0.5) / 0.5, the [-1, 1]
range the vggface2 checkpoint was trained with. Embeddings are L2-normalized
afterwards (cosine-style comparability, unit-bounded columns for XGBoost).

Why 512 raw columns and no PCA: a reduction would need FITTING, which would
turn the feature append into a stateful, split-dependent transform and break
the label-free cached-extraction contract that makes `supports_cv=True`
honest. The xgb-client matrix is ~116 encoded columns; 116 + 512 = 628
features on 5302 rows is comfortable for XGBoost, the unit-norm columns are
bounded, and the pre-registered expectation is a null — a reduction would
manufacture tuning freedom for nothing.

In [ ]:
import numpy as np

import arms_face  # module-level imports are numpy/pandas only — light, safe
from fraud_pipeline import (
    AXES,
    DEFAULT_RESULTS_PATH,
    load_enriched,
    load_results,
    print_comparison_table,
    register_arm,
    run_arm_on_axis,
)

# Canonical enriched frame + labels from the unified runner (the nb7/nb8 data
# section, shared by every arm — loaded once, used by everything below).
enriched, y = load_enriched()
customers = sorted(enriched['customer'].unique())
assert len(customers) == 100, 'expected 100 customers, got %d' % len(customers)
print(
    'fraud txns: %d of %d (%.2f%%) across %d unique customers'
    % (int(y.sum()), len(y), 100 * y.mean(), enriched['customer'].nunique())
)

# Generate-or-load in batches of 25: a killed cell (600 s ceiling) leaves a
# valid partial cache the next run continues from — never all-or-nothing.
BATCH_SIZE = 25
generated_total = 0
for start in range(0, len(customers), BATCH_SIZE):
    chunk = customers[start:start + BATCH_SIZE]
    flags = [arms_face.ensure_face_image(c)[1] for c in chunk]
    generated_total += sum(flags)
    print(
        'faces %3d-%3d: %d generated, %d loaded from cache'
        % (start + 1, start + len(chunk), sum(flags), len(chunk) - sum(flags))
    )
total_bytes = arms_face.assert_face_cache_within_budget()
print(
    'cache: %d JPEGs, %.1f MB (budget %d MB) — %d faces synthesized this run'
    % (len(list(arms_face.FACE_DIR.glob('*.jpg'))), total_bytes / 1e6,
       arms_face.FACE_CACHE_MAX_BYTES // (1024 * 1024), generated_total)
)

In [ ]:
# One FaceNet pass over the cache. embed_customers resolves per customer:
# process cache -> committed npz -> fresh FaceNet pass, and merges anything
# freshly computed back into data/face_embeddings.npz, so this cell is the
# single writer of the downstream artifact.
embeddings = arms_face.embed_customers(customers)
norms = np.linalg.norm(embeddings.to_numpy(), axis=1)
print(
    'embeddings: %s float32 | L2 norms in [%.6f, %.6f] (unit by construction)'
    % (embeddings.shape, norms.min(), norms.max())
)
npz_frame = arms_face.load_embeddings()
assert npz_frame is not None and set(customers).issubset(npz_frame.index), (
    'the committed npz must cover every customer after this cell'
)
print('committed npz:', arms_face.EMBEDDINGS_NPZ_PATH, npz_frame.shape)

In [ ]:
# Determinism check (decision D5). All comparisons are ALIGNED BY CUSTOMER
# ID — reindexed on the customer key, never positionally (the F3 review
# caught positional diffing misaligning two differently-ordered extraction
# runs and reporting a bogus 8.1e+01 "GPU nondeterminism" delta).
#
# 1. cache pass 2: generate-or-load must now produce 0 new files.
# 2. re-embed with BOTH caches bypassed (fresh FaceNet pass over the same
#    committed JPEGs) and compare against pass 1 on the customer key.
# 3. byte probe: regenerate ONE face off-cache and require the exact same
#    JPEG bytes as the cached image (same customer_id -> same z -> same
#    pixels), which is what anchors "same id => same image" empirically.
import tempfile
from pathlib import Path

pass2_generated = [arms_face.ensure_face_image(c)[1] for c in customers]
assert not any(pass2_generated), (
    'second cache pass must be pure load, %d generated' % sum(pass2_generated)
)
print('cache pass 2: 0 generated, %d loaded (generate-or-load is stable)' % len(customers))

emb_first = embeddings  # the pass-1 table from the cell above
emb_second = arms_face.embed_customers(customers, use_cache=False)
emb_second_aligned = emb_second.reindex(emb_first.index)
assert emb_second_aligned.index.equals(emb_first.index), 'customer keys must align'
assert list(emb_second_aligned.columns) == list(emb_first.columns), 'column order must align'
max_delta = float(np.max(np.abs(emb_second_aligned.to_numpy() - emb_first.to_numpy())))
print(
    'aligned-by-customer_id embedding re-check: max |delta| = %.3e -> %s'
    % (max_delta, 'bit-identical' if max_delta == 0.0 else 'GPU nondeterminism observed')
)
# The assert below is a misalignment guard, not a physics claim: identical
# eval-mode GPU convs reproduce bit-exactly, while a real misalignment or bug
# shows orders of magnitude larger than this tolerance.
assert max_delta < 1e-6, 'aligned embedding re-check failed: max |delta| = %.3e' % max_delta

with tempfile.TemporaryDirectory() as tmp_dir:
    probe = arms_face.generate_face_image(customers[0], Path(tmp_dir) / 'probe.jpg')
    same_bytes = probe.read_bytes() == arms_face.face_path(customers[0]).read_bytes()
print(
    'generation byte probe (%s): %s'
    % (customers[0], 'identical JPEG bytes' if same_bytes else 'MISMATCH')
)
assert same_bytes, 'same customer_id must synthesize byte-identical pixels (D5)'
print('D5 verdict: one customer_id -> one image -> one embedding, reproducible on this machine')

## The arm through the runner — all three axes

`run_arm_on_axis` owns the whole protocol per axis: axis split -> validation
carve -> model -> threshold frozen on the carve -> one-shot frozen-threshold
test evaluation + 1000-sample percentile-bootstrap AUC CIs. The arm is
registered IN-PROCESS here (the runner file is not modified by this
notebook), mirroring the timesfm-arm shape:

- `build_features` = the xgb-client matrix + the customer's 512-d face
  embedding broadcast to its rows; the lazy `arms_face` import inside
  `build_features` keeps the module chain importable without facenet/torch;
- `make_model` = the shared fixed-XGB factory (`fraud_pipeline.xgb_params`),
  so the arm's hypothesis — "does the face embedding add signal" — is held by
  keeping the model identical (timesfm-arm discipline);
- `supports_cv=True` because extraction is label-free and cached: the
  embedding depends ONLY on the customer id, fold membership never changes a
  feature value, so per-fold clones refit only the XGB. `cv=False` in this
  notebook — the evidence lives on the three test axes, which already
  exercise the cached extraction;
- `n_features` left unset — the runner counts the matrix itself
  (628 = 116 client-matrix columns + 512 face columns).

Zero label access is structural: `fraud_flag` never enters any expression —
the embedding is hash(customer_id) -> pixels -> FaceNet, nothing else.

The axes, in registry order:

- **random-grouped (PRIMARY)** — seeded random customer assignment,
  customer-disjoint, no time ordering;
- **grouped (stress)** — test = latest-seen customers;
- **chronological (stress)** — test strictly later than train.

In [ ]:
import fraud_pipeline as fp
from fraud_pipeline import ArmSkipped


def _build_face_features(enriched_frame):
    # xgb-client matrix + the customer's 512-d face embedding broadcast to its
    # rows (the timesfm-arm shape, fraud_pipeline.py:540-584). Lazy arms_face
    # import: the module chain must import without facenet/torch installed.
    import arms_face  # lazy

    missing = arms_face.check_dependencies()
    if missing is not None:
        raise ArmSkipped('face-features: missing dependency (%s)' % missing)
    enriched_face = arms_face.append_features(enriched_frame)
    return pd.concat(
        [fp._features_xgb_client(enriched_face), enriched_face[arms_face.FACE_FEATURES]],
        axis=1,
    )


def _make_face_model(y_fit):
    # The SAME fixed-XGB factory as xgb-client — the arm's hypothesis is
    # "does the face embedding add signal", held by keeping the model
    # identical (timesfm-arm discipline).
    from xgboost import XGBClassifier  # lazy: keeps module import light

    return XGBClassifier(**fp.xgb_params(y_fit))


register_arm(
    'face-features',
    'base+client features + per-customer StyleGAN2-face FaceNet embedding '
    '(nb13 arm); one 512-d unit-norm embedding per customer broadcast to its '
    'rows, zero label access — extraction cached, so CV refits only the XGB',
    make_model=_make_face_model,
    build_features=_build_face_features,
    supports_cv=True,
)

rows = []
for axis in AXES:  # registry order: random-grouped (PRIMARY), grouped, chronological
    rows.append(run_arm_on_axis('face-features', axis, enriched, y, cv=False))
print_comparison_table(rows, title='face-features — test metrics per axis (frozen threshold)')

## Ethical caveat — read this before reusing anything above

Face-to-fraud scoring is ethically dubious, and this notebook is NOT a
recommendation to build one:

- **Protected attributes.** A face embedding correlates with age, gender and
  ethnicity proxies; a fraud score keyed on it would discriminate on
  appearance with disparate impact and no ex-ante individual justification.
- **Appearance discrimination.** Even a statistically "useful" face signal
  would subject people to decisions based on how they look — indefensible in
  regulated decisioning regardless of measured performance.
- **Synthetic faces do not fix the mechanism.** StyleGAN2 images avoid harm
  to real people, but the proxy logic that would deploy is identical;
  synthetic data measures the pipeline, not the ethics.
- **What this actually is.** A methods-comparability experiment with a
  pre-registered null expectation (plan §7): does an arbitrary per-customer
  constant feature block change the arm ranking the runner reports? The
  correct production decision for this feature family is "do not ship".

## Face arm vs xgb-client — primary axis only

The F4 family question head to head: does the face embedding match or move
the best per-transaction arm under the identical protocol on the identical
split? The xgb-client row comes from the accumulated results file
(`results/fraud_pipeline_results.jsonl`, latest row per axis x arm — the same
superseding convention the runner's table uses; if the row is missing the
cell re-runs the arm through the pipeline instead of guessing). Both rows
carry test positives, chance level and bootstrap CIs — with ~13 test
positives on the primary axis the honest benchmark is distance-from-chance
plus interval overlap, not the point estimate alone.

In [ ]:
def latest_ok(axis, arm):
    # Latest ok row per (axis, arm) from the accumulated JSONL (nb12 helper).
    matches = [
        r
        for r in load_results(DEFAULT_RESULTS_PATH)
        if r.get('axis') == axis and r.get('arm') == arm and r.get('status') == 'ok'
    ]
    return matches[-1] if matches else None


face_primary = next(r for r in rows if r['axis'] == 'random-grouped')
xgb_row = latest_ok('random-grouped', 'xgb-client')
if xgb_row is None:  # results file unavailable/stale — re-run through the pipeline
    xgb_row = run_arm_on_axis('xgb-client', 'random-grouped', enriched, y, cv=False)

print_comparison_table(
    [face_primary, xgb_row],
    title='face-features vs xgb-client — random-grouped (PRIMARY axis)',
)


def covers_chance(row):
    return row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']


# Per-row verdicts: test positives and chance level printed NEXT to the
# metrics — 11-24 test positives make the absolute numbers unreadable alone.
for row in rows + [xgb_row]:
    print(
        '%-24s test positives %2d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
        % (
            row['axis'], row['test_positives'], row['pr_auc'],
            row['pr_auc_ci_low'], row['pr_auc_ci_high'], row['chance_level'],
            row['pr_auc'] - row['chance_level'],
            'covers chance' if covers_chance(row) else 'separates',
        )
    )

### Interpretation (read after the tables — null results are findings)

- **Noise budget first.** 91 frauds total; the three splits put 13
  (random-grouped), 11 (grouped) and 24 (chronological) test positives in
  play — printed on every verdict line next to its chance level. At that
  size a single swapped fraud moves test PR-AUC by hundredths and the
  bootstrap CIs span a wide band: deltas under ~0.05 PR-AUC between arms are
  noise, not signal.
- **Read every number against its chance level.** Chance PR-AUC equals the
  test positive rate; each verdict line above prints the row's delta and
  whether its bootstrap interval covers chance. An interval covering chance
  means the arm is indistinguishable from a random ranking on that split,
  whatever the point estimate suggests — a null here is a finding, not a
  failure.
- **What the face CAN and CANNOT do, per axis.** The embedding is a constant
  per customer. On random-grouped and grouped, test customers are disjoint
  from train: the model saw 80 faces, and the ~20 unseen test faces are just
  arbitrary unit-norm vectors — the pre-registered prediction is
  chance-level, and that null is the expected result. On chronological, most
  test customers appeared in train, so their EXACT face rides along: any
  lift there is the identity-memorization channel nb7 measured with client
  features — leakage by design, evidence about identity transfer, never
  about appearance.
- **The comparison that calibrates the runner.** The face arm is xgb-client
  plus 512 columns that are (a) pure per-customer constants, (b) mostly
  unseen at test time on the grouped axes. If it tracks xgb-client on the
  grouped axes, XGBoost's gain splits simply ignored the constant block; if
  it moves only on chronological, the movement is identity leakage. Either
  outcome measures how much an arbitrary high-dimensional per-customer block
  can distort the unified arm comparison — the methods point of F4.
- **Zero label access, restated.** The embedding is hash(customer_id) -> z ->
  StyleGAN2 pixels -> FaceNet, L2-normalized; `fraud_flag` is never read and
  fold membership cannot change a value — that is why `supports_cv=True` is
  honest and one cached extraction serves all three axes and every CV fold.
- **Determinism verdict as observed.** The check above runs the cache twice,
  re-embeds with both caches bypassed, compares the two passes ALIGNED BY
  CUSTOMER ID (reindexed on the customer key — never positionally, the F3
  review lesson), and byte-probes one regenerated face against its cached
  JPEG. The printed max |delta| and byte-identity verdict are the record,
  not an assumption.
- **The direction, not the score, is the finding.** A per-customer face is
  at best a customer-identity proxy: on customer-disjoint axes it cannot
  transfer, on chronological it can only leak identity. The committed
  artifact for downstream consumers (nb15, the runner) is
  `data/face_embeddings.npz` (100 x 512 float32 + customer_id + the seed
  rule), loadable without a GPU pass.

## Summary

- `src/arms_face.py` implements the `face-features` arm: one StyleGAN2 face
  per customer (FFHQ pkl; latent = SHA-256("face-arm-v1:" + customer_id)[:8]
  read big-endian -> `RandomState.randn(z_dim)`, truncation 0.7, const noise,
  force fp32), embedded by the vggface2 InceptionResnetV1 into 512
  L2-normalized dims, cached as JPEG under `data/faces/` (512 px long edge,
  <= 50 MB asserted) plus the committed npz
  `data/face_embeddings.npz` (100 x 512 float32 + customer_id).
- The arm is registered IN-PROCESS by this notebook — `fraud_pipeline.py` is
  not modified: xgb-client matrix + the customer's 512-d embedding broadcast
  to its rows, the shared fixed-XGB factory, `supports_cv=True` (extraction
  is label-free and cached), `n_features` counted by the runner (628).
- Determinism (D5) checked in-notebook: cache pass 2 loads 0 new files; two
  full embedding passes compared ALIGNED BY CUSTOMER ID (reindexed on the
  customer key); one face regenerated off-cache must reproduce the exact
  cached JPEG bytes.
- No network anywhere: thispersondoesnotexist was rejected as
  non-reproducible; the StyleGAN2 repo, FFHQ pkl and FaceNet checkpoint are
  all machine-local caches.
- Pre-registered null (plan §7): appearance carries no fraud signal — the
  per-customer face is at best a customer-identity proxy, untestable on the
  customer-grouped axes and pure identity leakage on chronological. The
  verdict lines above record exactly what was observed.
- Ethical position, restated: methods-comparability experiment with a
  pre-registered null, not a recommendation — face-based fraud scoring must
  not ship.